## LLM init

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="ollama_chat/llama3.1", temperature=0.0)

llm.invoke("Hi").content

## Pydantic Model

In [1]:
from pydantic import BaseModel, Field


class Conversation(BaseModel):
    question: str = Field(description="question from user")
    ai_response: str = Field(description="LLM response")

# Output Parsers

Output parsers are used to transform the unstructured text output of language models (LLMs) into structured formats like JSON or Pydantic models, making it easier to use the output in downstream tasks. This is helpful when you need to generate structured data from LLMs, normalize their outputs, or provide specific formatting instructions to the model through prompts.

## Pydantic parser

In [2]:
from langchain_core.output_parsers import PydanticOutputParser


pydantic_parser = PydanticOutputParser(pydantic_object=Conversation)

pydantic_parser.get_format_instructions()

C:\Users\argroy\arg_venv\Lib\site-packages\langchain_core\_api\deprecation.py:26: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"question": {"description": "question from user", "title": "Question", "type": "string"}, "ai_response": {"description": "LLM response", "title": "Ai Response", "type": "string"}}, "required": ["question", "ai_response"]}\n```'

In [3]:
llm_output = '''
```json
{
  "question": "What is iron man\'s first movie called?",
  "ai_response": "Iron Man (2008)"
}
```
'''

pydantic_output = pydantic_parser.invoke(llm_output)

print(pydantic_output)

question="What is iron man's first movie called?" ai_response='Iron Man (2008)'


In [4]:
print(isinstance(pydantic_output, BaseModel))

True


### Pydantic parsing using llm's output

In [6]:
from langchain_core.prompts import PromptTemplate


template = """
Answer the following user question to best of your knowledge.
Response format instructions: {format_instructions}
User question: {user_query}
"""

prompt = PromptTemplate(
    template=template,
    input_variables=["user_query"],
    partial_variables={"format_instructions": pydantic_parser.get_format_instructions()},
)


complete_prompt = prompt.invoke({"user_query": "What is iron man's first movie called?"})
output = llm.invoke(complete_prompt)
pydantic_parser.invoke(output)

=== Prompt sent to LLM ===
text='\nAnswer the following user question to the best of your knowledge.\nResponse format instructions:\nThe output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"description": "Information about the requested movie.", "properties": {"title": {"description": "The official title of the movie.", "title": "Title", "type": "string"}, "year": {"description": "The year the movie was released.", "title": "Year", "type": "integer"}, "director": {"description": "The director of the movie.", "title": "Director", "type": "string"}}, "required": ["title", "year", "director"]}

ResponseError: model 'llama3.2' not found (status code: 404)

## JSON parser

In [ ]:
from langchain_core.output_parsers import JsonOutputParser

json_parser = JsonOutputParser(pydantic_object=Conversation)

json_parser.get_format_instructions()

In [ ]:
llm_output = '''
```json
{
  "question": "What is iron man\'s first movie called?",
  "ai_response": "Iron Man (2008)"
}
```
'''

json_output = json_parser.invoke(llm_output)

print(json_output)

In [ ]:
print(isinstance(json_output, dict))

### JSON parsing using llm's output

In [ ]:
from langchain_core.prompts import PromptTemplate


template = """
Answer the following user question to best of your knowledge.
Response format instructions: {format_instructions}
User question: {user_query}
"""

prompt = PromptTemplate(
    template=template,
    input_variables=["user_query"],
    partial_variables={"format_instructions": json_parser.get_format_instructions()},
)


complete_prompt = prompt.invoke({"user_query": "What is iron man's first movie called?"})
output = llm.invoke(complete_prompt)
json_parser.invoke(output)

### Try out following parsers
1. *XML parser*
2. *YAML parsers*

URL: [Langchain output parsers](https://python.langchain.com/docs/how_to/#output-parsers)

## How to fix parsing errors

In [ ]:
llm_output = """
{
  'question': 'What is iron man\'s first movie called?',
  'ai_response': 'Iron Man (2008)'
}
"""

try:
	json_parser.invoke(llm_output)
except Exception as e:
	print(e)

In [ ]:
from langchain.output_parsers import OutputFixingParser

new_parser = OutputFixingParser.from_llm(parser=json_parser, llm=llm)

new_parser_output = new_parser.parse(llm_output)
new_parser_output

In [ ]:
isinstance(new_parser_output, dict)

# Document Loaders

Document loaders are specialized components of LangChain that facilitate the access and conversion of data from diverse formats and sources into a standardized document object. This object typically comprises content and associated metadata, enabling seamless integration and processing within LangChain applications.

## PDF loader

Some of the important parameters of PyPDFLoader are:
1. extract_images
2. mode
3. extraction_mode etc

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "documents/git-cheat-sheet.pdf"

loader = PyPDFLoader(file_path)
pages = loader.load()

pages

In [ ]:
file_path = "documents/git-cheat-sheet.pdf"

loader = PyPDFLoader(file_path=file_path)

pages = []
for page in loader.lazy_load():
	pages.append(page)

pages

## CSV Loader

Some of the important parameters of CSVLoader are:
1. metadata_columns
2. csv_args
3. encoding etc

In [ ]:
from langchain_community.document_loaders.csv_loader import CSVLoader

file_path = "documents/employee_profiles.csv"

loader = CSVLoader(file_path=file_path)

docs = await loader.aload()

for record in docs:
    print(record)

In [ ]:
file_path = "documents/employee_profiles.csv"

loader = CSVLoader(
    file_path=file_path,
    csv_args={
        "delimiter": ",",
        "quotechar": '"',
        "fieldnames": ["Name", "EmpID", "Skills"],
    },
)


async for record in loader.alazy_load():
    print(record)

## Web Loader

In [ ]:
import bs4
from langchain_community.document_loaders import WebBaseLoader

page_url = "https://www.google.com"

loader = WebBaseLoader(web_paths=[page_url])
docs = []
async for doc in loader.alazy_load():
    docs.append(doc)

In [ ]:
len(docs)

In [ ]:
docs[0].metadata

In [ ]:
from pprint import pprint
pprint(docs[0].page_content)

### Explore other loaders
1. *HTML loader*
2. *JSON loader* etc

URL: [Langchain document loaders](https://python.langchain.com/docs/how_to/#document-loaders)

# LCEL (LangChain Expression Language)

LangChain Expression Language (LCEL) is a "minimalist" code layer within LangChain, a framework for building applications powered by large language models (LLMs). It simplifies the process of creating and connecting various components of LangChain chains, making it easier to build complex workflows. 

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("tell me an intresting fact about {topic}")

chain = prompt | llm | StrOutputParser()

In [ ]:
response = chain.invoke({"topic": "Earth"})

from pprint import pprint
pprint(response)

URL: [LCEL](https://python.langchain.com/docs/how_to/#langchain-expression-language-lcel)

### .pipe()

In [ ]:
chain = (
    prompt
    .pipe(llm)
    .pipe(StrOutputParser())
)

response = chain.invoke({"topic": "Earth"})

from pprint import pprint
pprint(response)

### Gothrough the following types of Runnables
1. [RunnableSequence](https://python.langchain.com/api_reference/core/runnables/langchain_core.runnables.base.RunnableSequence.html)
2. [RunnableParallel](https://python.langchain.com/api_reference/core/runnables/langchain_core.runnables.base.RunnableParallel.html)
3. [RunnablePassthrough](https://python.langchain.com/api_reference/core/runnables/langchain_core.runnables.passthrough.RunnablePassthrough.html)

